# Backends: one Patcher, five `Field` adapters

`geotoolz.patch` keeps the locality algebra (`Geometry`, `Sampler`,
`Window`, `Aggregation`) decoupled from the substrate via the `Field` /
`Domain` Protocols. The same `SpatialPatcher` therefore drives:

| Field | Domain | Backend |
|---|---|---|
| `RasterField` | `RasterDomain` (`GeoDataBase`) | `RasterioReader`, `GeoTensor`, `AsyncGeoTIFFReader` |
| `RioXarrayField` | `RasterDomain` | rioxarray `DataArray` |
| `XarrayField` | `GridDomain` | `xarray.DataArray` (non-raster) |
| `GeoPandasField` | `VectorDomain` or `PointDomain` | `geopandas.GeoDataFrame` |
| `XvecField` | `PointDomain` | `xvec.Dataset` |

Each adapter is gated behind an optional extra (`grid`, `vector`, `point`,
`xarray-raster`); install with `pip install 'geotoolz[patch-full]'`.

In [1]:
import html

import numpy as np
import rasterio
from georeader.geotensor import GeoTensor
from IPython.display import HTML


# Collapsed repr helper — wrap every backend's rich display in a <details>
# block so the reader expands the ones they want to inspect rather than
# scrolling past everything inlined. Falls back to the text repr inside a
# <pre> when the object has no `_repr_html_` (e.g. `GeoTensor`).
def collapsed(obj, summary: str = "click to inspect") -> HTML:
    body = (
        obj._repr_html_()
        if hasattr(obj, "_repr_html_")
        else "<pre style='margin:0'>" + html.escape(repr(obj)) + "</pre>"
    )
    return HTML(
        f"<details><summary><b>{html.escape(summary)}</b></summary>{body}</details>"
    )


from geotoolz.patch import (
    Patch,
    RasterField,
    SpatialBoxcar,
    SpatialByIndex,
    SpatialExplicit,
    SpatialKNNGraph,
    SpatialOverlapAdd,
    SpatialPatcher,
    SpatialRadiusGraph,
    SpatialRectangular,
    SpatialRegularStride,
)

## 1. `RasterField` — `GeoTensor` / `RasterioReader`

The canonical raster backend. Wraps anything satisfying georeader's
`GeoData` Protocol — `GeoTensor` (in-memory), `RasterioReader` (lazy
file-backed), `AsyncGeoTIFFReader` (lazy async). The Patcher consumes them
identically.

In [2]:
arr = np.outer(np.linspace(0, 1, 32), np.linspace(0, 1, 32)).astype(np.float32)
print(f"raster arr.shape: {arr.shape}")  # (32, 32)
gt = GeoTensor(values=arr, transform=rasterio.Affine.identity(), crs="EPSG:32630")

raster arr.shape: (32, 32)


Underlying `GeoTensor` (numpy subclass + CRS/affine):

In [3]:
collapsed(gt, summary="GeoTensor")

In [4]:
raster_field = RasterField(gt)
print(f"raster_field.domain.shape: {raster_field.domain.shape}")  # (32, 32)
print(f"raster_field.domain.crs:   {raster_field.domain.crs}")

raster_patcher = SpatialPatcher(
    geometry=SpatialRectangular(size=(8, 8)),
    sampler=SpatialRegularStride(step=8),
    window=SpatialBoxcar(),
    aggregation=SpatialOverlapAdd(),
)
raster_patches = list(raster_patcher.split(raster_field))
print(f"raster: {len(raster_patches)} patches")
print(f"  first.indices: {raster_patches[0].indices}")
print(f"  first.data.shape: {raster_patches[0].data.shape}")

raster_field.domain.shape: (32, 32)
raster_field.domain.crs:   EPSG:32630
raster: 16 patches
  first.indices: Window(col_off=0, row_off=0, width=8, height=8)
  first.data.shape: (8, 8)


## 2. `RioXarrayField` — rioxarray-flavoured `DataArray`

Same raster domain, accessed through the `xarray` surface (chunked Dask
reads, unified xarray pipelines). The Patcher sees the affine + shape via
the rioxarray accessor and treats it identically to a `RasterioReader`.

In [5]:
import xarray as xr


# Default-collapse the inner sections of xarray's HTML repr so users have
# to click into Coordinates / Indexes / Attributes / Data themselves.
xr.set_options(
    display_expand_data=False,
    display_expand_attrs=False,
    display_expand_coords=False,
    display_expand_data_vars=False,
    display_expand_indexes=False,
)

from geotoolz.patch import RioXarrayField


xr_raster = xr.DataArray(
    arr[None, :, :],  # (band, y, x) — rioxarray convention
    dims=("band", "y", "x"),
    coords={
        "band": [1],
        "y": np.linspace(31.5, 0.5, 32),
        "x": np.linspace(0.5, 31.5, 32),
    },
)
xr_raster = xr_raster.rio.write_crs("EPSG:32630")
xr_raster = xr_raster.rio.write_transform(rasterio.Affine.identity())

Underlying `xarray.DataArray` with the rioxarray `.rio` accessor:

In [6]:
collapsed(xr_raster, summary="xarray.DataArray (rioxarray-flavoured)")

In [7]:
rio_xr_field = RioXarrayField(xr_raster)
print(
    f"rio_xr_field.domain.shape: {rio_xr_field.domain.shape}, "
    f"transform: {rio_xr_field.domain.transform}"
)

rio_xr_field.domain.shape: (1, 32, 32), transform: | 1.00, 0.00, 0.00|
| 0.00, 1.00, 31.00|
| 0.00, 0.00, 1.00|


## 3. `XarrayField` — non-raster N-D grid

For dense, labeled cubes that aren't necessarily raster — climate
reanalyses, model output. The Patcher dispatches `SpatialRectangular`
on `GridDomain`, slicing through `xarray.DataArray.isel`.

In [8]:
from geotoolz.patch import XarrayField


# Tiny (time=5, lat=64, lon=64) climate-like cube
cube = xr.DataArray(
    np.random.default_rng(0).standard_normal((5, 64, 64)).astype(np.float32),
    dims=("time", "lat", "lon"),
    coords={
        "time": np.arange(5),
        "lat": np.linspace(-90, 90, 64),
        "lon": np.linspace(-180, 180, 64),
    },
    name="temperature_anomaly",
)

Underlying `xarray.DataArray` (labeled coords, named dims, rich repr):

In [9]:
collapsed(cube, summary="xarray.DataArray (time × lat × lon)")

In [10]:
grid_field = XarrayField(cube)
print(f"grid_field.domain.shape: {grid_field.domain.shape}")
print(f"grid_field.domain.coords keys: {list(grid_field.domain.coords)}")

grid_patcher = SpatialPatcher(
    geometry=SpatialRectangular(size=(5, 16, 16)),  # full time × 16 lat × 16 lon
    sampler=SpatialRegularStride(step=(5, 16, 16)),
    window=SpatialBoxcar(),
    aggregation=SpatialByIndex(),  # ragged-friendly
)
grid_patches = list(grid_patcher.split(grid_field))
print(f"grid: {len(grid_patches)} patches")
print(f"  first.anchor: {grid_patches[0].anchor}")
print(f"  first.indices: {grid_patches[0].indices}")
print(f"  first.data.da.shape: {grid_patches[0].data.da.shape}")

grid_field.domain.shape: (5, 64, 64)
grid_field.domain.coords keys: ['time', 'lat', 'lon']
grid: 16 patches
  first.anchor: {'time': 0, 'lat': 0, 'lon': 0}
  first.indices: {'time': slice(0, 5, None), 'lat': slice(0, 16, None), 'lon': slice(0, 16, None)}
  first.data.da.shape: (5, 16, 16)


## 4. `GeoPandasField` — vector geometries

Polygons (or any vector geometry) over a `VectorDomain`. The `SpatialKNNGraph`
/ `SpatialRadiusGraph` geometries dispatch on the domain's spatial index
to find geometries near each anchor.

In [11]:
import geopandas as gpd
import shapely

from geotoolz.patch import GeoPandasField


# Synthetic admin polygons on a 10×10 grid
polygons = []
for i in range(8):
    for j in range(8):
        polygons.append(shapely.box(j * 1.0, i * 1.0, (j + 1) * 1.0, (i + 1) * 1.0))
gdf = gpd.GeoDataFrame(
    {"id": np.arange(len(polygons)), "geometry": polygons},
    crs="EPSG:32630",
)

Underlying `geopandas.GeoDataFrame` (pandas DataFrame + a geometry column):

In [12]:
collapsed(gdf.head(8), summary="geopandas.GeoDataFrame (polygons)")

,id,geometry
0,0,"POLYGON ((1 0, 1 1, 0 1, 0 0, 1 0))"
1,1,"POLYGON ((2 0, 2 1, 1 1, 1 0, 2 0))"
2,2,"POLYGON ((3 0, 3 1, 2 1, 2 0, 3 0))"
3,3,"POLYGON ((4 0, 4 1, 3 1, 3 0, 4 0))"
4,4,"POLYGON ((5 0, 5 1, 4 1, 4 0, 5 0))"
5,5,"POLYGON ((6 0, 6 1, 5 1, 5 0, 6 0))"
6,6,"POLYGON ((7 0, 7 1, 6 1, 6 0, 7 0))"
7,7,"POLYGON ((8 0, 8 1, 7 1, 7 0, 8 0))"


In [13]:
vector_field = GeoPandasField(gdf)
print(f"vector_field.domain.crs: {vector_field.domain.crs}")
print(f"vector_field.domain.bounds: {vector_field.domain.bounds}")

# Find all polygons within radius 1.5 of the centre of the grid
vector_patcher = SpatialPatcher(
    geometry=SpatialRadiusGraph(radius=1.5),
    sampler=SpatialExplicit(anchors_=[shapely.Point(4.0, 4.0)]),
    window=SpatialBoxcar(),
    aggregation=SpatialByIndex(),
)
vector_patches = list(vector_patcher.split(vector_field))
print(f"vector: {len(vector_patches)} patches")
print(f"  first.indices length: {len(vector_patches[0].indices)}")
print(f"  matching polygon IDs: {vector_patches[0].data.gdf['id'].tolist()}")

vector_field.domain.crs: EPSG:32630
vector_field.domain.bounds: (0.0, 0.0, 8.0, 8.0)
vector: 1 patches
  first.indices length: 16
  matching polygon IDs: [18, 19, 20, 27, 28, 21, 26, 42, 29, 34, 35, 36, 37, 43, 44, 45]


## 5. `GeoPandasField(as_points=True)` — point cloud via geopandas

When every geometry is a `shapely.Point`, ask for a `PointDomain` view —
the field builds a `cKDTree` over the point coordinates so `KNNGraph` /
`RadiusGraph` queries are cheap.

In [14]:
rng = np.random.default_rng(0)
pts = rng.uniform(-1, 1, size=(80, 2))
pt_gdf = gpd.GeoDataFrame(
    {"value": rng.standard_normal(80)},
    geometry=gpd.points_from_xy(pts[:, 0], pts[:, 1]),
    crs="EPSG:32630",
)

Underlying point-only `geopandas.GeoDataFrame` (same shape, different
geometry type — the `as_points=True` flag tells the adapter to expose a
`PointDomain` with a `cKDTree` instead of a `VectorDomain`):

In [15]:
collapsed(pt_gdf.head(6), summary="geopandas.GeoDataFrame (points)")

,value,geometry
0,-0.106072,POINT (0.274 -0.46)
1,-1.185720,POINT (-0.918 -0.967)
2,-2.398233,POINT (0.627 0.826)
3,0.513052,POINT (0.213 0.459)
4,-0.297584,POINT (0.087 0.87)
5,-0.530008,POINT (0.632 -0.995)


In [16]:
point_field = GeoPandasField(pt_gdf, as_points=True)
print(f"point_field.domain.coords.shape: {point_field.domain.coords.shape}")
print(f"point_field.domain has kdtree: {point_field.domain.kdtree is not None}")

point_patcher = SpatialPatcher(
    geometry=SpatialKNNGraph(k=5),
    sampler=SpatialExplicit(anchors_=[np.array([0.0, 0.0])]),
    window=SpatialBoxcar(),
    aggregation=SpatialByIndex(),
)
point_patches = list(point_patcher.split(point_field))
print(f"point: {len(point_patches)} patches")
print(f"  k-NN neighbours: {point_patches[0].indices.tolist()}")
print(f"  data type: {type(point_patches[0].data).__name__}")

point_field.domain.coords.shape: (80, 2)
point_field.domain has kdtree: True
point: 1 patches
  k-NN neighbours: [37, 65, 70, 12, 60]
  data type: GeoPandasField


## 6. `XvecField` — xvec data cubes for in-situ multivariate data

`xvec` puts a `shapely.Point` geometry coordinate on an `xarray.Dataset`,
which is the modern pattern for stations / floats / swath samples with
multiple variables and times.

In [17]:
import xvec  # noqa: F401  — registers the .xvec accessor

from geotoolz.patch import XvecField


# Synthetic xvec dataset: 30 stations, 24 hourly measurements each
n_stations, n_hours = 30, 24
station_xy = rng.uniform(-1, 1, size=(n_stations, 2))
station_geoms = gpd.points_from_xy(station_xy[:, 0], station_xy[:, 1])
xvec_ds = xr.Dataset(
    {
        "temperature": (
            ("geometry", "time"),
            rng.standard_normal((n_stations, n_hours)),
        ),
        "pressure": (
            ("geometry", "time"),
            1000 + rng.standard_normal((n_stations, n_hours)),
        ),
    },
    coords={"geometry": station_geoms, "time": np.arange(n_hours)},
).xvec.set_geom_indexes("geometry", crs="EPSG:32630")

Underlying `xarray.Dataset` with an xvec-registered `geometry` coord
(multivariate stations × time data cube):

In [18]:
collapsed(xvec_ds, summary="xarray.Dataset (xvec, stations × time)")

In [19]:
xvec_field = XvecField(xvec_ds)
print(f"xvec_field.domain.coords.shape: {xvec_field.domain.coords.shape}")

xvec_patcher = SpatialPatcher(
    geometry=SpatialKNNGraph(k=4),
    sampler=SpatialExplicit(anchors_=[np.array([0.0, 0.0])]),
    window=SpatialBoxcar(),
    aggregation=SpatialByIndex(),
)
xvec_patches = list(xvec_patcher.split(xvec_field))
print(f"xvec: {len(xvec_patches)} patches")
print(f"  k-NN neighbours: {xvec_patches[0].indices.tolist()}")
print(f"  data.ds.dims: {dict(xvec_patches[0].data.ds.sizes)}")

xvec_field.domain.coords.shape: (30, 2)
xvec: 1 patches
  k-NN neighbours: [25, 6, 26, 11]
  data.ds.dims: {'geometry': 4, 'time': 24}


## A common operator

Same `SpatialPatcher` shape, different backends. The composition algebra
(`GridSampler` → `ApplyToChips` → `Stitch`) doesn't care which substrate
it's running on — the operator just sees `patch.data` in whatever shape the
backend produces.

In [20]:
from geotoolz.core import Lambda


def _patch_data_shape(d) -> str:
    """Walk a few common substrate-specific attrs to find a shape-like value."""
    if hasattr(d, "shape"):
        return repr(tuple(d.shape))
    for attr in ("da", "gdf", "ds"):
        sub = getattr(d, attr, None)
        if sub is not None and hasattr(sub, "shape"):
            return f"<{attr}> {tuple(sub.shape)}"
        if sub is not None and hasattr(sub, "sizes"):
            return f"<{attr}> dims={dict(sub.sizes)}"
    return f"<no shape: {type(d).__name__}>"


def _summarise(p: Patch) -> dict:
    return {
        "anchor": repr(p.anchor)[:40],
        "data_type": type(p.data).__name__,
        "data_shape": _patch_data_shape(p.data),
    }


_label = Lambda(_summarise, name="summary")

for name, patches in [
    ("RasterField", raster_patches),
    ("XarrayField (grid)", grid_patches),
    ("GeoPandasField (vector)", vector_patches),
    ("GeoPandasField (points)", point_patches),
    ("XvecField", xvec_patches),
]:
    summary = _label(patches[0])
    print(f"{name:>26s}: {summary}")

               RasterField: {'anchor': '(0, 0)', 'data_type': 'GeoTensor', 'data_shape': '(8, 8)'}
        XarrayField (grid): {'anchor': "{'time': 0, 'lat': 0, 'lon': 0}", 'data_type': 'XarrayField', 'data_shape': '<da> (5, 16, 16)'}
   GeoPandasField (vector): {'anchor': '<POINT (4 4)>', 'data_type': 'GeoPandasField', 'data_shape': '<gdf> (16, 2)'}
   GeoPandasField (points): {'anchor': 'array([0., 0.])', 'data_type': 'GeoPandasField', 'data_shape': '<gdf> (5, 2)'}
                 XvecField: {'anchor': 'array([0., 0.])', 'data_type': 'XvecField', 'data_shape': "<ds> dims={'geometry': 4, 'time': 24}"}


## What this proves

- The four-axis Patcher composition is substrate-agnostic. Same
  `Sampler` / `Window` / `Aggregation` triples work across every Field;
  only the `Geometry` dispatches on the `Domain` type.
- Adding a new substrate is a single ~30-line `Field` adapter — implement
  `domain`, `select(indexer)`, `with_data(array)`, gate the optional
  dependency with a friendly import error.
- Choice of backend is a deployment concern, not a modelling one. The same
  pipeline that runs against an in-memory `GeoTensor` in a notebook runs
  against a lazy `RasterioReader` in production with no code change beyond
  the field wrapper.